<a href="https://colab.research.google.com/github/sareena151/lost-luggage-detective/blob/main/lost_luggage_detective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
print("Lost Luggage Detective & Routing System Initialized")

Lost Luggage Detective & Routing System Initialized


In [2]:
# ---------- Scan record (Linked List ka NODE) ----------
class ScanNode:
    def __init__(self, location, scan_type, time):
        self.location = location      # e.g. "Karachi Check-in"
        self.scan_type = scan_type    # check-in, sorting, loading...
        self.time = time              # e.g. "08:00"
        self.next = None              # agla scan (pointer)


# ---------- Bag ----------
class Bag:
    def __init__(self, barcode, owner, flight_id, planned_route):
        self.barcode = barcode
        self.owner = owner
        self.flight_id = flight_id
        self.planned_route = planned_route   # list, e.g. ["Karachi","Dubai","London"]
        self.head = None                     # history ki shuruaat (Linked List head)
        self.tail = None                     # history ka aakhri scan


# ---------- Flight ----------
class Flight:
    def __init__(self, flight_id, source, destination, departure):
        self.flight_id = flight_id
        self.source = source
        self.destination = destination
        self.departure = departure

In [3]:
bag = Bag("BG-784512", "Ali Khan", "PK-201", ["Karachi", "Dubai", "London"])
print(bag.barcode, bag.planned_route)

BG-784512 ['Karachi', 'Dubai', 'London']


In [4]:
class LuggageSystem:
    def __init__(self):
        self.bags = {}      # HASH TABLE: barcode -> Bag
        self.flights = {}   # HASH TABLE: flight_id -> Flight

    # ---------- Flight add karna ----------
    def add_flight(self, flight):
        self.flights[flight.flight_id] = flight

    # ---------- Bag register karna ----------
    def register_bag(self, barcode, owner, flight_id, planned_route):
        if barcode in self.bags:                 # O(1) check
            print("Error: barcode already registered!")
            return None
        bag = Bag(barcode, owner, flight_id, planned_route)
        self.bags[barcode] = bag                 # O(1) insert
        return bag

    # ---------- Bag dhoondna ----------
    def find_bag(self, barcode):
        return self.bags.get(barcode)            # O(1) lookup

    # ---------- Scan add karna (LINKED LIST append) ----------
    def add_scan(self, barcode, location, scan_type, time):
        bag = self.find_bag(barcode)
        if bag is None:
            print("Error: bag not found!")
            return
        node = ScanNode(location, scan_type, time)
        if bag.head is None:       # pehla scan
            bag.head = node
            bag.tail = node
        else:                      # end mein jodo
            bag.tail.next = node
            bag.tail = node

    # ---------- History dikhana (LINKED LIST traversal) ----------
    def show_history(self, barcode):
        bag = self.find_bag(barcode)
        if bag is None:
            print("Bag not found!")
            return
        print(f"\nMovement history of {barcode} ({bag.owner}):")
        current = bag.head
        step = 1
        while current is not None:
            print(f"  {step}. [{current.time}] {current.location} - {current.scan_type}")
            current = current.next
            step += 1

In [5]:
system = LuggageSystem()

system.add_flight(Flight("PK-201", "Karachi", "Dubai", "10:00"))
system.add_flight(Flight("EK-501", "Dubai", "London", "14:00"))
system.add_flight(Flight("EK-702", "Dubai", "Doha", "13:30"))

system.register_bag("BG-784512", "Ali Khan", "PK-201", ["Karachi", "Dubai", "London"])

system.add_scan("BG-784512", "Karachi Check-in", "check-in", "08:00")
system.add_scan("BG-784512", "Karachi Sorting Hall", "sorting", "08:45")
system.add_scan("BG-784512", "Dubai Arrival", "arrival", "12:30")
system.add_scan("BG-784512", "Dubai Transfer Belt", "transfer", "13:00")
system.add_scan("BG-784512", "Dubai Gate (Doha flight EK-702)", "loading", "13:20")

system.show_history("BG-784512")


Movement history of BG-784512 (Ali Khan):
  1. [08:00] Karachi Check-in - check-in
  2. [08:45] Karachi Sorting Hall - sorting
  3. [12:30] Dubai Arrival - arrival
  4. [13:00] Dubai Transfer Belt - transfer
  5. [13:20] Dubai Gate (Doha flight EK-702) - loading


In [6]:
system.register_bag("BG-784512", "Someone", "PK-201", ["Karachi"])   # duplicate
system.show_history("BG-000000")                                      # galat barcode

Error: barcode already registered!
Bag not found!


In [7]:
from collections import deque
import re


class Tracer:
    def __init__(self, system):
        self.system = system
        self.graph = {}          # GRAPH (adjacency list): airport -> [airports]
        self.build_graph()

    # ---------- Flights se graph banana ----------
    def build_graph(self):
        self.graph = {}
        for f in self.system.flights.values():
            self.graph.setdefault(f.source, [])
            self.graph.setdefault(f.destination, [])
            self.graph[f.source].append(f.destination)

    def show_graph(self):
        print("Airport network:")
        for airport, neighbours in self.graph.items():
            print(f"  {airport} -> {neighbours}")

    # ---------- BFS (QUEUE use hoti hai) ----------
    def shortest_route(self, start, goal):
        queue = deque([[start]])      # queue mein paths rakhte hain
        visited = {start}
        while queue:
            path = queue.popleft()    # queue ka pehla path nikalo
            node = path[-1]
            if node == goal:
                return path
            for neighbour in self.graph.get(node, []):
                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append(path + [neighbour])
        return None                   # koi raasta nahi mila

    # ---------- Deviation detection ----------
    def detect_deviation(self, barcode):
        bag = self.system.find_bag(barcode)
        if bag is None:
            print("Bag not found!")
            return None
        last = bag.tail               # aakhri scan (O(1) kyunke tail rakha hai)
        if last is None:
            print("No scans recorded for this bag.")
            return None

        print(f"\nTracing bag {barcode}")
        print(f"Planned route : {' -> '.join(bag.planned_route)}")
        print(f"Last recorded : {last.location} at {last.time}")

        airport = last.location.split()[0]            # e.g. "Dubai"
        match = re.search(r"[A-Z]{2}-\d{3}", last.location)   # flight ID, e.g. EK-702

        if last.scan_type == "loading" and match and airport in bag.planned_route:
            flight = self.system.flights.get(match.group())   # hash table lookup
            idx = bag.planned_route.index(airport)
            expected_next = bag.planned_route[idx + 1] if idx + 1 < len(bag.planned_route) else None
            if flight and flight.destination != expected_next:
                print(f"DEVIATION DETECTED! Planned next stop was {expected_next}, "
                      f"but bag was loaded on {flight.flight_id} to {flight.destination}.")
                return True
        print("No deviation found.")
        return False

In [8]:
tracer = Tracer(system)

tracer.show_graph()

print("\nShortest route Karachi -> London:", tracer.shortest_route("Karachi", "London"))
print("Shortest route Karachi -> Doha  :", tracer.shortest_route("Karachi", "Doha"))
print("Shortest route London -> Karachi:", tracer.shortest_route("London", "Karachi"))

tracer.detect_deviation("BG-784512")

Airport network:
  Karachi -> ['Dubai']
  Dubai -> ['London', 'Doha']
  London -> []
  Doha -> []

Shortest route Karachi -> London: ['Karachi', 'Dubai', 'London']
Shortest route Karachi -> Doha  : ['Karachi', 'Dubai', 'Doha']
Shortest route London -> Karachi: None

Tracing bag BG-784512
Planned route : Karachi -> Dubai -> London
Last recorded : Dubai Gate (Doha flight EK-702) at 13:20
DEVIATION DETECTED! Planned next stop was London, but bag was loaded on EK-702 to Doha.


True

In [9]:
def to_minutes(t):
    h, m = t.split(":")
    return int(h) * 60 + int(m)


class Ranker:
    def __init__(self, system):
        self.system = system

    # ---------- SORTING: Bubble Sort (score ke hisaab se, bada pehle) ----------
    def sort_desc(self, items):
        n = len(items)
        for i in range(n - 1):
            for j in range(n - 1 - i):
                if items[j]["score"] < items[j + 1]["score"]:
                    items[j], items[j + 1] = items[j + 1], items[j]   # swap
        return items

    # ---------- Next locations suggest karna ----------
    def suggest_next(self, barcode):
        bag = self.system.find_bag(barcode)          # hash table lookup
        if bag is None or bag.tail is None:
            print("Bag not found or no scans recorded!")
            return None

        last = bag.tail                               # aakhri scan
        airport = last.location.split()[0]            # e.g. "Dubai"
        last_time = to_minutes(last.time)
        match = re.search(r"[A-Z]{2}-\d{3}", last.location)
        loaded_flight = match.group() if match else None

        candidates = []
        for f in self.system.flights.values():
            if f.source != airport:                   # sirf is airport se nikalne wali flights
                continue
            score = 0
            notes = []

            if f.flight_id == loaded_flight:
                score += 50
                notes.append("bag was last loaded on this flight")

            if f.destination in bag.planned_route:
                score += 20
                notes.append("on planned route")
                if loaded_flight and f.flight_id != loaded_flight:
                    notes.append("possible only if the loading scan was an entry error")
            else:
                notes.append("POSSIBLE MISROUTE (outside planned route)")

            gap = to_minutes(f.departure) - last_time
            if 0 <= gap <= 30:
                score += 15
                notes.append(f"departs {gap} min after last scan")
            elif 30 < gap <= 60:
                score += 8
                notes.append(f"departs {gap} min after last scan")

            candidates.append({"place": f"{f.destination} arrival (flight {f.flight_id})",
                               "score": score, "notes": notes})

        # Lost & Found hamesha fallback
        candidates.append({"place": f"{airport} Lost & Found",
                           "score": 30, "notes": ["fallback, always included"]})

        ranked = self.sort_desc(candidates)

        print(f"\nPossible next locations for {barcode} (highest priority first):")
        for i, c in enumerate(ranked, start=1):
            print(f"  {i}. {c['place']} - score {c['score']}")
            for n in c["notes"]:
                print(f"       - {n}")
        return ranked

In [10]:
ranker = Ranker(system)
result = ranker.suggest_next("BG-784512")


Possible next locations for BG-784512 (highest priority first):
  1. Doha arrival (flight EK-702) - score 65
       - bag was last loaded on this flight
       - POSSIBLE MISROUTE (outside planned route)
       - departs 10 min after last scan
  2. Dubai Lost & Found - score 30
       - fallback, always included
  3. London arrival (flight EK-501) - score 28
       - on planned route
       - possible only if the loading scan was an entry error
       - departs 40 min after last scan


In [11]:
class Corrector:
    def __init__(self, system, tracer):
        self.system = system
        self.tracer = tracer
        self.undo_stack = []        # STACK: corrections ka record (LIFO)

    # ---------- Linked list se aakhri node hatana ----------
    def remove_last_scan(self, bag):
        if bag.head is None:
            return
        if bag.head is bag.tail:            # sirf ek node tha
            bag.head = None
            bag.tail = None
            return
        current = bag.head
        while current.next is not bag.tail: # second-last node tak jao
            current = current.next
        current.next = None
        bag.tail = current

    # ---------- Correction + re-route ----------
    def correct_and_reroute(self, barcode, airport, flight_id, found_time, load_time):
        bag = self.system.find_bag(barcode)
        flight = self.system.flights.get(flight_id)
        if bag is None or flight is None:
            print("Error: bag or flight not found!")
            return
        if flight.source != airport:
            print("Error: this flight does not leave from", airport)
            return

        final_destination = bag.planned_route[-1]
        new_route = self.tracer.shortest_route(airport, final_destination)   # BFS
        if new_route is None or flight.destination != new_route[1]:
            print("Error: this flight does not lead towards", final_destination)
            return

        self.system.add_scan(barcode, f"{airport} Recovery Desk", "correction", found_time)
        self.system.add_scan(barcode, f"{airport} Gate ({flight.destination} flight {flight_id})",
                             "loading", load_time)
        self.undo_stack.append(barcode)                                     # STACK push

        print(f"\nCorrection recorded for {barcode}")
        print("New route:", " -> ".join(new_route))
        print(f"Bag re-loaded on {flight_id} ({airport} -> {flight.destination})")

    # ---------- Undo (STACK pop) ----------
    def undo_last_correction(self):
        if not self.undo_stack:
            print("Nothing to undo.")
            return
        barcode = self.undo_stack.pop()                                     # STACK pop
        bag = self.system.find_bag(barcode)
        self.remove_last_scan(bag)       # loading scan hatao
        self.remove_last_scan(bag)       # correction scan hatao
        print(f"Last correction undone for {barcode}")

In [12]:
corrector = Corrector(system, tracer)

corrector.correct_and_reroute("BG-784512", "Dubai", "EK-501", "13:40", "13:50")
system.show_history("BG-784512")
result = tracer.detect_deviation("BG-784512")

print("\n--- UNDO TEST ---")
corrector.undo_last_correction()
system.show_history("BG-784512")
result = tracer.detect_deviation("BG-784512")


Correction recorded for BG-784512
New route: Dubai -> London
Bag re-loaded on EK-501 (Dubai -> London)

Movement history of BG-784512 (Ali Khan):
  1. [08:00] Karachi Check-in - check-in
  2. [08:45] Karachi Sorting Hall - sorting
  3. [12:30] Dubai Arrival - arrival
  4. [13:00] Dubai Transfer Belt - transfer
  5. [13:20] Dubai Gate (Doha flight EK-702) - loading
  6. [13:40] Dubai Recovery Desk - correction
  7. [13:50] Dubai Gate (London flight EK-501) - loading

Tracing bag BG-784512
Planned route : Karachi -> Dubai -> London
Last recorded : Dubai Gate (London flight EK-501) at 13:50
No deviation found.

--- UNDO TEST ---
Last correction undone for BG-784512

Movement history of BG-784512 (Ali Khan):
  1. [08:00] Karachi Check-in - check-in
  2. [08:45] Karachi Sorting Hall - sorting
  3. [12:30] Dubai Arrival - arrival
  4. [13:00] Dubai Transfer Belt - transfer
  5. [13:20] Dubai Gate (Doha flight EK-702) - loading

Tracing bag BG-784512
Planned route : Karachi -> Dubai -> Londo

In [13]:
# ---------- Demo data (proposal ka scenario) ----------
def load_demo_data():
    s = LuggageSystem()
    s.add_flight(Flight("PK-201", "Karachi", "Dubai", "10:00"))
    s.add_flight(Flight("EK-501", "Dubai", "London", "14:00"))
    s.add_flight(Flight("EK-702", "Dubai", "Doha", "13:30"))

    s.register_bag("BG-784512", "Ali Khan", "PK-201", ["Karachi", "Dubai", "London"])
    s.add_scan("BG-784512", "Karachi Check-in", "check-in", "08:00")
    s.add_scan("BG-784512", "Karachi Sorting Hall", "sorting", "08:45")
    s.add_scan("BG-784512", "Dubai Arrival", "arrival", "12:30")
    s.add_scan("BG-784512", "Dubai Transfer Belt", "transfer", "13:00")
    s.add_scan("BG-784512", "Dubai Gate (Doha flight EK-702)", "loading", "13:20")
    return s


# ---------- MENU ----------
def run_menu():
    system = load_demo_data()
    tracer = Tracer(system)
    ranker = Ranker(system)
    corrector = Corrector(system, tracer)

    while True:
        print("\n===== LOST LUGGAGE DETECTIVE & ROUTING SYSTEM =====")
        print("1. Register bag")
        print("2. Add scan")
        print("3. Search bag")
        print("4. Show movement history")
        print("5. Report missing bag (trace)")
        print("6. Correct and re-route")
        print("7. Undo last correction")
        print("8. Show airport network and routes")
        print("0. Exit")
        choice = input("Enter choice: ").strip()

        if choice == "1":
            barcode = input("Barcode (e.g. BG-100001): ").strip()
            owner = input("Owner name: ").strip()
            flight_id = input("Flight ID: ").strip()
            route = [x.strip() for x in input("Planned route (comma separated, e.g. Karachi,Dubai,London): ").split(",")]
            if system.register_bag(barcode, owner, flight_id, route):
                print("Bag registered successfully.")

        elif choice == "2":
            barcode = input("Barcode: ").strip()
            scan_type = input("Scan type (check-in/sorting/arrival/transfer/loading): ").strip().lower()
            if scan_type == "loading":
                airport = input("Airport (e.g. Dubai): ").strip()
                fid = input("Flight ID loaded onto: ").strip()
                flight = system.flights.get(fid)
                if flight is None:
                    print("Error: flight not found!")
                    continue
                location = f"{airport} Gate ({flight.destination} flight {fid})"
            else:
                location = input("Location (e.g. Dubai Arrival): ").strip()
            time = input("Time (HH:MM): ").strip()
            system.add_scan(barcode, location, scan_type, time)
            print("Scan recorded.")

        elif choice == "3":
            barcode = input("Barcode: ").strip()
            bag = system.find_bag(barcode)
            if bag is None:
                print("Bag not found!")
            else:
                print(f"Found: {bag.barcode} | Owner: {bag.owner} | Flight: {bag.flight_id}")
                print("Planned route:", " -> ".join(bag.planned_route))

        elif choice == "4":
            system.show_history(input("Barcode: ").strip())

        elif choice == "5":
            barcode = input("Barcode of missing bag: ").strip()
            system.show_history(barcode)
            tracer.detect_deviation(barcode)
            ranker.suggest_next(barcode)

        elif choice == "6":
            barcode = input("Barcode: ").strip()
            airport = input("Airport where bag was found (e.g. Dubai): ").strip()
            fid = input("Correct flight ID (e.g. EK-501): ").strip()
            found_time = input("Time bag was found (HH:MM): ").strip()
            load_time = input("Time of re-loading (HH:MM): ").strip()
            corrector.correct_and_reroute(barcode, airport, fid, found_time, load_time)

        elif choice == "7":
            corrector.undo_last_correction()

        elif choice == "8":
            tracer.build_graph()
            tracer.show_graph()
            start = input("Route from: ").strip()
            goal = input("Route to: ").strip()
            route = tracer.shortest_route(start, goal)
            print("Route:", " -> ".join(route) if route else "No route found")

        elif choice == "0":
            print("Goodbye!")
            break

        else:
            print("Invalid choice, try again.")

In [21]:
def load_demo_data():
    s = LuggageSystem()
    s.add_flight(Flight("PK-201", "Karachi", "Dubai", "10:00"))
    s.add_flight(Flight("EK-501", "Dubai", "London", "14:00"))
    s.add_flight(Flight("EK-702", "Dubai", "Doha", "13:30"))

    s.register_bag("BG-784512", "Ali Khan", "PK-201", ["Karachi", "Dubai", "London"])
    s.add_scan("BG-784512", "Karachi Check-in", "check-in", "08:00")
    s.add_scan("BG-784512", "Karachi Sorting Hall", "sorting", "08:45")
    s.add_scan("BG-784512", "Dubai Arrival", "arrival", "12:30")
    s.add_scan("BG-784512", "Dubai Transfer Belt", "transfer", "13:00")
    s.add_scan("BG-784512", "Dubai Gate (Doha flight EK-702)", "loading", "13:20")
    return s

print("load_demo_data ready")

load_demo_data ready


In [23]:
run_menu()


===== LOST LUGGAGE DETECTIVE & ROUTING SYSTEM =====
1. Register bag
2. Add scan
3. Search bag
4. Show movement history
5. Report missing bag (trace)
6. Correct and re-route
7. Undo last correction
8. Show airport network and routes
0. Exit
Enter choice: 4
Barcode: BG-784512

Movement history of BG-784512 (Ali Khan):
  1. [08:00] Karachi Check-in - check-in
  2. [08:45] Karachi Sorting Hall - sorting
  3. [12:30] Dubai Arrival - arrival
  4. [13:00] Dubai Transfer Belt - transfer
  5. [13:20] Dubai Gate (Doha flight EK-702) - loading

===== LOST LUGGAGE DETECTIVE & ROUTING SYSTEM =====
1. Register bag
2. Add scan
3. Search bag
4. Show movement history
5. Report missing bag (trace)
6. Correct and re-route
7. Undo last correction
8. Show airport network and routes
0. Exit
Enter choice: 5
Barcode of missing bag: BG-784512

Movement history of BG-784512 (Ali Khan):
  1. [08:00] Karachi Check-in - check-in
  2. [08:45] Karachi Sorting Hall - sorting
  3. [12:30] Dubai Arrival - arrival
  4.